In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv


# Project overview

The dataset for this competition (both train and test) was inspired by F1 strategy dataset. Feature distributions are close to, but not exactly the same, as the original, and we intentionally remove Normalized_TyreLife which makes the prediction trivial. Feel free to use the original dataset as part of this competition, both to explore differences as well as to see whether incorporating the original in training improves model performance.

Files
train.csv - the training set, with PitNextLap as target
test.csv - the test set, used to predict the likelihood for PitNextLap
sample_submission.csv - a sample submission file in the correct format

# Import libraries

In [2]:
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import linear, relu, sigmoid
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder 

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import make_pipeline

2026-05-10 13:29:42.894420: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778419783.037849      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778419783.080422      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778419783.422519      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778419783.422554      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778419783.422556      16 computation_placer.cc:177] computation placer alr

# Load and inspect data

In [3]:
df=pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
df.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [5]:
df.isna().sum()

id                        0
Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

# preprocess

## Define X and y

In [7]:
# Create the input feature matrix
X = df.drop(["PitNextLap", "id"], axis=1)

# Create the target vector
y = df ["PitNextLap"]

In [8]:
print ('The shape of X is: ' + str(X.shape))
print ('The shape of y is: ' + str(y.shape))

The shape of X is: (439140, 14)
The shape of y is: (439140,)


In [9]:
y.value_counts()

PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64

## Split data 

In [10]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Preprocessing pipeline

In [11]:
# Create the preprocessing pipeline for categorical data
cat_selector =  X_train.select_dtypes('object').columns
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_pipe = make_pipeline(ohe)

cat_tuple = ('categorical',cat_pipe, cat_selector)
cat_tuple 
# define numeric features and object features
num_cols = X_train.select_dtypes("number").columns

# feature scaling 
scaler = StandardScaler()  

# Instantiate the pipeline
num_pipe = make_pipeline(scaler)

# Make the tuple for ColumnTransformer
num_tuple = ('numeric',num_pipe,num_cols)
num_tuple
# Create the preprocessing ColumnTransformer
preprocessor = ColumnTransformer([cat_tuple, num_tuple],
verbose_feature_names_out=False)

preprocessor

ColumnTransformer(transformers=[('categorical',
                                 Pipeline(steps=[('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 Index(['Driver', 'Compound', 'Race'], dtype='object')),
                                ('numeric',
                                 Pipeline(steps=[('standardscaler',
                                                  StandardScaler())]),
                                 Index(['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change'],
      dtype='object'))],
                  verbose_feature_names_out=False)

In [12]:
preprocessor.fit(X_train)
x_train_transformed=preprocessor.transform(X_train)
x_test_transformed=preprocessor.transform(X_test)

In [13]:
from imblearn.over_sampling import SMOTE as smote
smote=smote(random_state=42)
X_train_smote,y_train_smote=smote.fit_resample(x_train_transformed,y_train)


# Build NN model

In [14]:
from tensorflow.keras import regularizers
model = tf.keras.Sequential([
    tf.keras.layers.Dense(
        64, activation='relu',
        input_shape=(912,),
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.Dropout(0.3),          
    tf.keras.layers.Dense(
        32, activation='relu',
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-05-10 13:32:03.952166: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [15]:

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',    
    metrics=['accuracy']
)


## Train model

In [16]:
print("x_train_transformed shape:", X_train_smote.shape)
print("y_train shape:", y_train_smote.shape)
print("y_test shape:", y_test.shape)

x_train_transformed shape: (562946, 912)
y_train shape: (562946,)
y_test shape: (87828,)


In [17]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,    
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train_smote, y_train_smote,
    epochs=10,               
    validation_data=(x_test_transformed, y_test),
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/10
17593/17593 ━━━━━━━━━━━━━━━━━━━━ 41s 2ms/step - accuracy: 0.8367 - loss: 0.4032 - val_accuracy: 0.8594 - val_loss: 0.3419
Epoch 2/10
17593/17593 ━━━━━━━━━━━━━━━━━━━━ 39s 2ms/step - accuracy: 0.8700 - loss: 0.3457 - val_accuracy: 0.8621 - val_loss: 0.3457
Epoch 3/10
17593/17593 ━━━━━━━━━━━━━━━━━━━━ 40s 2ms/step - accuracy: 0.8718 - loss: 0.3405 - val_accuracy: 0.8602 - val_loss: 0.3342
Epoch 4/10
17593/17593 ━━━━━━━━━━━━━━━━━━━━ 42s 2ms/step - accuracy: 0.8743 - loss: 0.3367 - val_accuracy: 0.8552 - val_loss: 0.3430
Epoch 5/10
17593/17593 ━━━━━━━━━━━━━━━━━━━━ 40s 2ms/step - accuracy: 0.8741 - loss: 0.3375 - val_accuracy: 0.8590 - val_loss: 0.3411
Epoch 6/10
17593/17593 ━━━━━━━━━━━━━━━━━━━━ 40s 2ms/step - accuracy: 0.8752 - loss: 0.3356 - val_accuracy: 0.8519 - val_loss: 0.3571
Epoch 7/10
17593/17593 ━━━━━━━━━━━━━━━━━━━━ 40s 2ms/step - accuracy: 0.8764 - loss: 0.3338 - val_accuracy: 0.8577 - val_loss: 0.3381
Epoch 8/10
17593/17593 ━━━━━━━━━━━━━━━━━━━━ 39s 2ms/step - accuracy: 

In [18]:

pred_probs = model.predict(x_test_transformed)

if pred_probs.ndim == 2 and pred_probs.shape[1] == 1:
    pred_probs = pred_probs.squeeze()

pred = (pred_probs > 0.5).astype(int)

print('Predicted:', pred[:5])
print('True:', y_train_smote[:5])


2745/2745 ━━━━━━━━━━━━━━━━━━━━ 2s 770us/step
Predicted: [0 0 0 1 0]
True: 0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: PitNextLap, dtype: float64


# Test data

In [19]:
df2=pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/test.csv')
new_x = preprocessor.transform(df2)


In [20]:
pred_probs = model.predict(new_x)        
if pred_probs.ndim == 2 and pred_probs.shape[1] == 1:
    pred_probs = pred_probs.squeeze()

pred = (pred_probs > 0.5).astype(int)


tensorflow = pd.DataFrame({
    'id': df2['id'].values,
    'PitNextLap': pred
})

tensorflow.to_csv('/kaggle/working/sample_submission.csv', index=False)

5881/5881 ━━━━━━━━━━━━━━━━━━━━ 4s 751us/step


In [21]:
df3 =pd.read_csv('/kaggle/working/sample_submission.csv')
df3.head()

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,1
